## 00 — Map Events

Every map we have built so far has been **static from Python's perspective** — you run the cell, the map appears, and Python is done. The user can pan and zoom, but nothing feeds back into your code.

This lesson changes that. ipyleaflet maps generate **events** — structured messages sent to Python every time the user does something. You write a function that receives those messages and responds to them.

```
Static map    →  user sees map
Interactive map  →  user asks map questions  →  Python responds
```

## What Is an Interactive Map?

You already know what interactive maps feel like:

- Clicking a location on Google Maps drops a pin and shows its address
- Clicking a country on a news graphic shows a popup with data
- Drawing a route measures the total distance

In every case, **a user action triggered a response**. The map noticed where you clicked and did something with that information.

In ipyleaflet, *you* write the response. The map tells you what happened; your Python function decides what to do with it.

## The ipyleaflet Event System

ipyleaflet uses an **observer pattern**:

1. You write a handler function
2. You register it with the map using `m.on_interaction(handler)`
3. Every time the user interacts with the map, ipyleaflet calls your handler with a dict of event data

```
user action  →  ipyleaflet  →  your handler function  →  your response
```

The handler always has the same signature: it accepts `**kwargs`.

In [ ]:
from ipyleaflet import Map

WICHITA_FALLS = (33.9137, -98.4934)

m = Map(center=WICHITA_FALLS, zoom=12)

# This handler prints every piece of data the event sends
def handle_interaction(**kwargs):
    print(kwargs)

m.on_interaction(handle_interaction)
m

Click anywhere on the map above. You will see a dict printed below the cell — that is the raw event payload.

Notice that **every mouse movement** also fires an event. That is a lot of output. In the next section we will filter to only the events we care about.

## The Event Data Structure

Every event dict contains at least a `"type"` key. Click events also include `"coordinates"`. The full structure looks like this:

```python
{
    "type":        "click",          # event type string
    "coordinates": [lat, lon],       # where the click landed
}
```

The `"type"` field tells you what kind of interaction occurred. The common types are:

| `type` value | When it fires |
|---|---|
| `"click"` | single left-click |
| `"dblclick"` | double-click |
| `"mousemove"` | cursor moves across map |
| `"mousedown"` | mouse button pressed |
| `"mouseup"` | mouse button released |
| `"contextmenu"` | right-click |

Most of the time you only care about `"click"`. Filter by checking `kwargs["type"]` at the top of your handler.

In [ ]:
from ipyleaflet import Map

m = Map(center=WICHITA_FALLS, zoom=12)

def handle_click(**kwargs):
    # Ignore everything except actual clicks
    if kwargs.get("type") != "click":
        return

    print(kwargs)  # inspect the full payload

m.on_interaction(handle_click)
m

Now only clicks produce output. Notice the `"coordinates"` key — it holds `[latitude, longitude]` in ipyleaflet order.

## Extracting Coordinates

Pull `lat` and `lon` out of the coordinates list.

In [ ]:
from ipyleaflet import Map

m = Map(center=WICHITA_FALLS, zoom=12)

def handle_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    print(f"lat={lat:.5f}  lon={lon:.5f}")

m.on_interaction(handle_click)
m

## A Coordinate Logger

Collecting clicked coordinates into a list turns the map into a **coordinate generator** — an extremely useful tool for building GeoJSON by hand, marking targets, or creating test data.

Each click appends to `clicked_points`. The list grows as you click.

In [ ]:
from ipyleaflet import Map

m = Map(center=WICHITA_FALLS, zoom=12)

clicked_points = []

def log_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    clicked_points.append((lat, lon))
    print(f"[{len(clicked_points)}]  lat={lat:.5f}  lon={lon:.5f}")

m.on_interaction(log_click)
m

In [ ]:
# Inspect the collected points after clicking
print(f"{len(clicked_points)} points collected:")
for i, (lat, lon) in enumerate(clicked_points):
    print(f"  {i+1}.  ({lat:.5f}, {lon:.5f})")

## Placing a Marker on Click

The logger is useful on its own, but the real power is **changing the map in response to a click**. Here we drop a marker at every clicked location.

In [ ]:
from ipyleaflet import Map, Marker

m = Map(center=WICHITA_FALLS, zoom=12)

def place_marker(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    marker = Marker(location=(lat, lon), title=f"{lat:.4f}, {lon:.4f}")
    m.add(marker)

m.on_interaction(place_marker)
m

Click several locations. Each click immediately adds a marker — no cell re-run needed. This is the widget model from Lesson 01 in action: the map is a live object, and `m.add()` updates it in place.

## Exercise A

Modify the click handler to only place a marker on **every other click** — odd-numbered clicks (1st, 3rd, 5th...) get a marker; even-numbered clicks are silently ignored. Print which clicks are skipped.

In [ ]:
from ipyleaflet import Map, Marker

m = Map(center=WICHITA_FALLS, zoom=12)
markers = []

# Modify the handler to only place a marker on every OTHER click (1st, 3rd, 5th...)
# Even-numbered clicks are silently ignored
# Your code here

m

## Exercise B

Extend your click handler to handle two event types:

- **Single click** (`"click"`) — drops a default marker
- **Double-click** (`"dblclick"`) — removes all markers from the map and prints `"Cleared"`

Tip: check `kwargs.get("type")` for both values.

In [ ]:
from ipyleaflet import Map, Marker, AwesomeIcon

m = Map(center=WICHITA_FALLS, zoom=12)

# Single click → default blue marker
# Double-click ("dblclick") → clear all markers
# Your code here

m

---

## Check Your Understanding

Build a map that:

1. Drops a marker on every click
2. Prints the total number of markers placed after each click (e.g., `"3 markers placed"`)

```python
# your answer here
```

## Next

In [01 — Click Interactions](./01-Click_Interactions.ipynb), we use click coordinates to build things — numbered markers, growing paths, and a GeoJSON export of everything collected.